In [20]:
# Import necessary libraries for data manipulation and modeling
import pandas as pd
import numpy as np
import lightgbm as lgb
import catboost as cb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error
from sklearn.impute import SimpleImputer
from preprocessing import preprocess_f1_data
from sklearn.metrics import roc_auc_score, precision_score
import warnings
warnings.filterwarnings('ignore')

In [21]:
# Load the F1 Strategy Dataset
df = pd.read_csv('data/f1_strategy_dataset_v4.csv')

# Display the first few rows
df.head()

,Driver,LapNumber,Compound,Stint,TyreLife,Position,LapTime (s),Race,Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change
0,ALB,1,MEDIUM,1,2.0,17,100.625,Abu Dhabi Grand Prix,2023,0.000,0.000,0,0,0.017241,0.117647,0.0
1,ALB,2,MEDIUM,1,3.0,18,93.560,Abu Dhabi Grand Prix,2023,-7.065,-7.065,0,0,0.034483,0.176471,-1.0
2,ALB,3,MEDIUM,1,4.0,18,91.768,Abu Dhabi Grand Prix,2023,-1.792,-8.857,0,0,0.051724,0.235294,0.0
3,ALB,4,MEDIUM,1,5.0,18,91.591,Abu Dhabi Grand Prix,2023,-0.177,-9.034,0,0,0.068966,0.294118,0.0
4,ALB,5,MEDIUM,1,6.0,18,91.422,Abu Dhabi Grand Prix,2023,-0.169,-9.203,0,0,0.086207,0.352941,0.0


In [22]:
# Basic statistics
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101371 entries, 0 to 101370
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Driver                  101371 non-null  object 
 1   LapNumber               101371 non-null  int64  
 2   Compound                101305 non-null  object 
 3   Stint                   101371 non-null  int64  
 4   TyreLife                101371 non-null  float64
 5   Position                101371 non-null  int64  
 6   LapTime (s)             101371 non-null  float64
 7   Race                    101371 non-null  object 
 8   Year                    101371 non-null  int64  
 9   LapTime_Delta           101371 non-null  float64
 10  Cumulative_Degradation  101371 non-null  float64
 11  PitStop                 101371 non-null  int64  
 12  PitNextLap              101371 non-null  int64  
 13  RaceProgress            101371 non-null  float64
 14  Normalized_TyreLife 

In [23]:
df.describe()

,LapNumber,Stint,TyreLife,Position,LapTime (s),Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change
count,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000
mean,30.444841,2.046394,14.549339,9.759132,92.587188,2023.589685,-0.203891,-29.550051,0.251581,0.254797,0.432618,0.386521,-0.004636
std,18.146942,0.948797,10.313385,5.406456,33.231414,1.098518,45.344910,70.235759,0.433924,0.435749,0.258129,0.259906,3.912725
min,1.000000,1.000000,1.000000,1.000000,67.012000,2022.000000,-2403.895000,-274.564000,0.000000,0.000000,0.012821,0.012821,-18.000000
25%,15.000000,1.000000,7.000000,5.000000,82.021000,2023.000000,-7.253000,-51.054500,0.000000,0.000000,0.210526,0.172414,-2.000000
50%,30.000000,2.000000,13.000000,10.000000,91.167000,2024.000000,-0.027000,-21.678000,0.000000,0.000000,0.421053,0.333333,0.000000
75%,45.000000,3.000000,20.000000,14.000000,99.356000,2025.000000,5.705000,-3.725500,1.000000,1.000000,0.631579,0.562500,2.000000
max,78.000000,8.000000,78.000000,20.000000,2526.253000,2025.000000,2433.472000,2412.431000,1.000000,1.000000,1.000000,1.000000,18.000000


In [24]:
# Target distribution
df['PitNextLap'].value_counts()
df['PitNextLap'].value_counts(normalize=True)

PitNextLap
0    0.745203
1    0.254797
Name: proportion, dtype: float64

In [25]:
# Missing values
df.isnull().sum()

Driver                     0
LapNumber                  0
Compound                  66
Stint                      0
TyreLife                   0
Position                   0
LapTime (s)                0
Race                       0
Year                       0
LapTime_Delta              0
Cumulative_Degradation     0
PitStop                    0
PitNextLap                 0
RaceProgress               0
Normalized_TyreLife        0
Position_Change            0
dtype: int64

In [26]:
# Time-series aware train/test split

# Separate data by year
train_df = df[df['Year'].isin([2022, 2023])].copy()
val_df = df[df['Year'] == 2024].copy()
test_df = df[df['Year'] == 2025].copy()

print(f"Total rows: {len(df)}")
print(f"Train size: {len(train_df)} ({len(train_df)/len(df):.2%})")
print(f"Val size: {len(val_df)} ({len(val_df)/len(df):.2%})")
print(f"Test size: {len(test_df)} ({len(test_df)/len(df):.2%})")
print(f"\nTrain/Val/Test ratio: {len(train_df)/len(val_df):.2f} / {len(train_df)/len(test_df):.2f}")

# Verify temporal order is preserved
print(f"\nUnique sequences in train: {train_df.groupby(['Driver', 'Race', 'Year']).ngroups}")
print(f"Unique sequences in val: {val_df.groupby(['Driver', 'Race', 'Year']).ngroups}")
print(f"Unique sequences in test: {test_df.groupby(['Driver', 'Race', 'Year']).ngroups}")

Total rows: 101371
Train size: 46774 (46.14%)
Val size: 27557 (27.18%)
Test size: 27040 (26.67%)

Train/Val/Test ratio: 1.70 / 1.73

Unique sequences in train: 878
Unique sequences in val: 485
Unique sequences in test: 485


In [27]:
# Create lagged features for Position, LapTime, and degradation metrics
# Group by Driver, Race, Year to maintain temporal order within sequences

# Create lagged features
train_df['Position_lag1'] = train_df.groupby(['Year', 'Race', 'Driver'])['Position'].shift(1)
train_df['Position_lag2'] = train_df.groupby(['Year', 'Race', 'Driver'])['Position'].shift(2)
train_df['Position_lag3'] = train_df.groupby(['Year', 'Race', 'Driver'])['Position'].shift(3)

# LapTime lags
train_df['LapTime_lag1'] = train_df.groupby(['Year', 'Race', 'Driver'])['LapTime (s)'].shift(1)
train_df['LapTime_lag2'] = train_df.groupby(['Year', 'Race', 'Driver'])['LapTime (s)'].shift(2)
train_df['LapTime_lag3'] = train_df.groupby(['Year', 'Race', 'Driver'])['LapTime (s)'].shift(3)

# Rolling average pace (using LapTime_Delta as a proxy for pace change)
# Rolling average of LapTime_Delta over 3 laps
train_df['Rolling_avg_pace'] = train_df.groupby(['Year', 'Race', 'Driver'])['LapTime_Delta'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

# Rolling degradation (rolling average of Cumulative_Degradation)
train_df['Rolling_degradation'] = train_df.groupby(['Year', 'Race', 'Driver'])['Cumulative_Degradation'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

# Fill NaN values (from shifts) with 0
# Only fill numeric columns with 0, leave string/categorical columns unchanged
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
train_df[numeric_cols] = train_df[numeric_cols].fillna(0)


In [28]:
# Convert Compound column to categorical with specified order
categories = ['SOFT', 'MEDIUM', 'HARD']
dtype = pd.CategoricalDtype(categories=categories, ordered=True)
train_df['Compound'] = train_df['Compound'].astype(dtype)

# Verify the conversion
train_df['Compound'].cat.categories.tolist()
train_df['Compound'].cat.codes.value_counts()

 2    19612
 1    17357
 0     7349
-1     2456
Name: count, dtype: int64

In [29]:
# Impute missing values
imputer = SimpleImputer(strategy='most_frequent')
train_df['Compound'] = imputer.fit_transform(train_df['Compound'].values.reshape(-1, 1)).flatten()

In [30]:
# Encode Compound column with OrdinalEncoder
encoder = OrdinalEncoder()
train_df['Compound_encoded'] = encoder.fit_transform(train_df['Compound'].to_numpy().reshape(-1, 1)).flatten()

# Drop the original Compound column
train_df.drop('Compound', axis=1, inplace=True)

In [31]:
train_df.head()

,Driver,LapNumber,Stint,TyreLife,Position,LapTime (s),Race,Year,LapTime_Delta,Cumulative_Degradation,...,Position_Change,Position_lag1,Position_lag2,Position_lag3,LapTime_lag1,LapTime_lag2,LapTime_lag3,Rolling_avg_pace,Rolling_degradation,Compound_encoded
0,ALB,1,1,2.0,17,100.625,Abu Dhabi Grand Prix,2023,0.000,0.000,...,0.0,0.0,0.0,0.0,0.000,0.000,0.000,0.000000,0.000000,1.0
1,ALB,2,1,3.0,18,93.560,Abu Dhabi Grand Prix,2023,-7.065,-7.065,...,-1.0,17.0,0.0,0.0,100.625,0.000,0.000,-3.532500,-3.532500,1.0
2,ALB,3,1,4.0,18,91.768,Abu Dhabi Grand Prix,2023,-1.792,-8.857,...,0.0,18.0,17.0,0.0,93.560,100.625,0.000,-2.952333,-5.307333,1.0
3,ALB,4,1,5.0,18,91.591,Abu Dhabi Grand Prix,2023,-0.177,-9.034,...,0.0,18.0,18.0,17.0,91.768,93.560,100.625,-3.011333,-8.318667,1.0
4,ALB,5,1,6.0,18,91.422,Abu Dhabi Grand Prix,2023,-0.169,-9.203,...,0.0,18.0,18.0,18.0,91.591,91.768,93.560,-0.712667,-9.031333,1.0


In [32]:
train_df = train_df.drop(['Driver', 'Race'], axis=1)

In [33]:
X_train, y_train = train_df.drop('PitNextLap', axis=1), train_df['PitNextLap']

In [34]:
# Train LightGBM model
lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)
lgb_model.fit(X_train, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 7752, number of negative: 39022
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0,003720 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2827
[LightGBM] [Info] Number of data points in the train set: 46774, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0,165733 -> initscore=-1,616175
[LightGBM] [Info] Start training from score -1,616175


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [35]:
cb_model = cb.CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=0
)
cb_model.fit(X_train, y_train)

CatBoostClassifier(depth=6, iterations=1000, learning_rate=0.05, random_seed=42, verbose=0)

In [36]:
# Preprocess validation data
X_val, y_val = preprocess_f1_data(val_df, imputer, encoder, is_training=False)

In [37]:
# Evaluate LightGBM model on validation set
y_pred_val = lgb_model.predict(X_val)
y_pred_proba_val = lgb_model.predict_proba(X_val)

print("\n=== Final Validation Results ===")
precision = precision_score(y_val, y_pred_val)
print(f"Precision: {precision:.4f}")
print(f"AUC-ROC: {roc_auc_score(y_val, y_pred_proba_val[:, 1]):.4f}")


=== Final Validation Results ===
Precision: 0.7826
AUC-ROC: 0.7605


In [38]:
# Evaluate CatBoost model on validation set
y_pred_val = cb_model.predict(X_val)
y_pred_proba_val = cb_model.predict_proba(X_val)

print("\n=== Final Validation Results ===")
precision = precision_score(y_val, y_pred_val)
print(f"Precision: {precision:.4f}")
print(f"AUC-ROC: {roc_auc_score(y_val, y_pred_proba_val[:, 1]):.4f}")


=== Final Validation Results ===
Precision: 0.8077
AUC-ROC: 0.6451
